# Part 1 — Scam-Aware RAG Chatbot (PH Anti-Scam & Cyber-Law Advisories)

**Purpose.** Online scams (phishing links, fake delivery/bank/e-wallet messages, "send me money" chats) are common in the Philippines, but the laws and advisories that protect victims are long and hard to read. This chatbot answers plain-language questions such as *"What counts as money muling?"* or *"What happens if I don't register my SIM?"* using **only** the official documents you load, and it shows the **source file and page** for every answer. If the documents do not contain the answer, it says so instead of guessing.

**Intended users.** Students, organization members, and ordinary users who want a quick, cited explanation. It is an information tool, **not legal advice**.

**Pipeline** (adapted from the reference notebook, with fixes):

`documents → clean → chunk (with page numbers) → MiniLM embeddings → cosine top-k → grounded prompt → flan-t5 → answer + citations`

Changes from the reference notebook: page-level citations; chunks sized so the retrieved context actually fits the 512-token window of flan-t5 (the earlier setup of 5 × 900-character chunks was silently truncated); a similarity threshold so out-of-scope questions are refused; and a proper evaluation (retrieval accuracy, latency, abstention, manual answer grading, chunk-size ablation).

**Part 2 (`Taglish_Scam_NER.ipynb`)** trains the NER model; Section 9 here connects the two.

In [ ]:
!pip -q install pymupdf sentence-transformers transformers sentencepiece accelerate scikit-learn pandas beautifulsoup4

## 1. Document source

Download **official** copies and upload them in the next cell. Keep the RA number in each file name (the evaluation matches on it). Save web pages as PDF (browser → Print → Save as PDF), or upload `.html` / `.txt`.

| Suggested file name | Document | Where to get it |
|---|---|---|
| `RA10173_DataPrivacyAct.pdf` | Data Privacy Act of 2012 | lawphil.net / officialgazette.gov.ph |
| `RA10175_CybercrimePreventionAct.pdf` | Cybercrime Prevention Act of 2012 | lawphil.net / officialgazette.gov.ph |
| `RA11934_SIMRegistrationAct.pdf` | SIM Registration Act | lawphil.net / officialgazette.gov.ph |
| `RA11765_FinancialConsumerProtection.pdf` | Financial Products and Services Consumer Protection Act | lawphil.net / officialgazette.gov.ph |
| `RA12010_AFASA.pdf` | Anti-Financial Account Scamming Act (approved July 20, 2024) | lawphil.net/statutes/repacts/ra2024/ra_12010_2024.html |
| `BSP_...`, `NPC_...`, `CICC_...` (optional) | Current scam advisories / consumer alerts | bsp.gov.ph, privacy.gov.ph, cicc.gov.ph |

Using the full texts plus 2-3 current advisories gives a few hundred chunks — enough to show real retrieval behaviour. State in your report the **date you downloaded** each document, since laws and advisories change.

In [ ]:
import os
from google.colab import files

DOCS_DIR = "docs"
os.makedirs(DOCS_DIR, exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    with open(os.path.join(DOCS_DIR, name), "wb") as f:
        f.write(data)
print("Files in docs/:", sorted(os.listdir(DOCS_DIR)))

Saving RA12010_AFASA.pdf to RA12010_AFASA.pdf
Saving RA11765_FinancialConsumerProtection.pdf to RA11765_FinancialConsumerProtection.pdf
Saving RA11934_SIMRegistrationAct.pdf to RA11934_SIMRegistrationAct.pdf
Saving RA10175_CybercrimePreventionAct.pdf to RA10175_CybercrimePreventionAct.pdf
Saving RA10173_DataPrivacyAct.pdf to RA10173_DataPrivacyAct (1).pdf
Files in docs/: ['RA10173_DataPrivacyAct (1).pdf', 'RA10173_DataPrivacyAct.pdf', 'RA10175_CybercrimePreventionAct.pdf', 'RA11765_FinancialConsumerProtection.pdf', 'RA11934_SIMRegistrationAct.pdf', 'RA12010_AFASA.pdf']


## 2. Load, clean and chunk (keeping page numbers)

In [ ]:
import re, time
import numpy as np
import pandas as pd
import fitz                      # PyMuPDF
from bs4 import BeautifulSoup

def load_pages(path):
    """Return [(page_label, text), ...]. PDFs keep real page numbers."""
    ext = path.lower().rsplit(".", 1)[-1]
    if ext == "pdf":
        pdf = fitz.open(path)
        pages = [(i + 1, p.get_text("text")) for i, p in enumerate(pdf)]
        pdf.close()
        return pages
    if ext in ("html", "htm"):
        soup = BeautifulSoup(open(path, encoding="utf-8", errors="replace").read(), "html.parser")
        for t in soup(["script", "style"]):
            t.decompose()
        return [("-", soup.get_text(" "))]
    if ext == "txt":
        return [("-", open(path, encoding="utf-8", errors="replace").read())]
    return []

def clean(text):
    return re.sub(r"\s+", " ", text).strip()

pages = []
for fname in sorted(os.listdir(DOCS_DIR)):
    for page_no, text in load_pages(os.path.join(DOCS_DIR, fname)):
        text = clean(text)
        if text:
            pages.append({"source": fname, "page": page_no, "text": text})

print("Documents:", len({p['source'] for p in pages}), "| pages/sections with text:", len(pages))
print("Sample:", pages[0]["text"][:300] if pages else "NO TEXT FOUND (scanned PDF? use OCR or another copy)")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Documents: 6 | pages/sections with text: 90
Sample: 1 Republic of the Philippines Congress of the Philippines Metro Manila Fifteenth Congress Second Regular Session Begun and held in Metro Manila, on Monday, the twenty-fifth day of July, two thousand eleven. [REPUBLIC ACT NO. 10173] AN ACT PROTECTING INDIVIDUAL PERSONAL INFORMATION IN INFORMATION AND


In [ ]:
def chunk_text(text, size=500, overlap=80):
    """Character chunks that end at a word boundary."""
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            sp = text.rfind(" ", start + int(size * 0.6), end)
            if sp != -1:
                end = sp
        c = text[start:end].strip()
        if len(c) >= 40:
            chunks.append(c)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def build_index(chunk_size=500, overlap=80):
    chunks = []
    for p in pages:
        for c in chunk_text(p["text"], chunk_size, overlap):
            chunks.append({"source": p["source"], "page": p["page"], "text": c})
    emb = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                          convert_to_numpy=True, batch_size=64, show_progress_bar=False)
    return {"chunks": chunks, "emb": emb}

INDEX = build_index(500, 80)
print("Chunks:", len(INDEX["chunks"]), "| embedding shape:", INDEX["emb"].shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunks: 510 | embedding shape: (510, 384)


In [ ]:
def retrieve(question, top_k=3, index=None):
    index = index or INDEX
    q = embedder.encode([question], normalize_embeddings=True, convert_to_numpy=True)[0]
    scores = index["emb"] @ q
    top = np.argsort(scores)[::-1][:top_k]
    return [{**index["chunks"][i], "score": float(scores[i])} for i in top]

for r in retrieve("What is money muling?"):
    print(f"{r['source']} p.{r['page']}  score={r['score']:.3f}\n  {r['text'][:200]}...\n")

RA12010_AFASA.pdf p.2  score=0.511
  Section 4. Prohibited Acts. - The following acts shall constitute Financial Account scamming under this Act: (a) Money Muling Activities. - A person performing any of the followings acts for the purpo...

RA12010_AFASA.pdf p.2  score=0.353
  r allowing the use of a Financial Account; (2) Opening a Financial Account under a fictitious name or using the identity or identification documents of another; (3) Buying or renting a Financial Accou...

RA12010_AFASA.pdf p.2  score=0.337
  funds. Section 7. Temporary Holding of Funds Subject of a Disputed Transaction. - Institutions shall have the authority to temporarily hold the funds subject of a disputed transaction within the perio...



## 3. Generator and the chatbot

`flan-t5-base` is used by default (runs on free Colab). Change `MODEL_NAME` to `google/flan-t5-large` for better answers (slower, more memory).

Design choices:
- **Token budget:** retrieved chunks are added to the prompt only while they fit in ~380 tokens, so nothing important is cut off by the 512-token input limit.
- **Abstention:** if the best similarity is below `MIN_SCORE`, the bot refuses without generating (calibrated in Section 5).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
generator = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device).eval()
print("Loaded", MODEL_NAME, "on", device)

MIN_SCORE = 0.30     # tuned in Section 5
NO_ANSWER = "I do not have enough information in the loaded documents to answer that."

def build_prompt(question, retrieved, max_ctx_tokens=380):
    parts, used = [], 0
    for r in retrieved:
        n = len(tokenizer.encode(r["text"], add_special_tokens=False))
        if parts and used + n > max_ctx_tokens:
            break
        parts.append(r["text"])
        used += n
    context = "\n\n".join(parts)
    prompt = (
        "Answer the question using only the context below. Give a short, direct answer. "
        "If the context does not contain the answer, say: \"I do not have enough information.\"\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    return prompt, len(parts)

def ask(question, top_k=3, min_score=None, index=None):
    min_score = MIN_SCORE if min_score is None else min_score
    t0 = time.perf_counter()
    retrieved = retrieve(question, top_k, index)
    t1 = time.perf_counter()
    if retrieved[0]["score"] < min_score:
        return {"answer": NO_ANSWER, "sources": retrieved, "abstained": True,
                "t_retrieve": t1 - t0, "t_total": t1 - t0}
    prompt, n_used = build_prompt(question, retrieved)
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)
    with torch.no_grad():
        out = generator.generate(**inputs, max_new_tokens=128, num_beams=4,
                                 no_repeat_ngram_size=3, early_stopping=True)
    answer = tokenizer.decode(out[0], skip_special_tokens=True).strip()
    t2 = time.perf_counter()
    abstained = "do not have enough information" in answer.lower()
    return {"answer": answer, "sources": retrieved[:max(n_used, 1)], "abstained": abstained,
            "t_retrieve": t1 - t0, "t_total": t2 - t0}

def show_answer(question, **kw):
    r = ask(question, **kw)
    print("Q:", question)
    print("A:", r["answer"])
    print("Sources:", "; ".join(f"{s['source']} p.{s['page']} ({s['score']:.2f})" for s in r["sources"]))
    print(f"Latency: retrieval {r['t_retrieve']*1000:.0f} ms | total {r['t_total']:.2f} s\n")

show_answer("What is a social engineering scheme?")
show_answer("What is the capital of Japan?")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded google/flan-t5-base on cpu
Q: What is a social engineering scheme?
A: a person who obtains ring Schemes
Sources: RA12010_AFASA.pdf p.2 (0.63); RA12010_AFASA.pdf p.2 (0.61); RA10173_DataPrivacyAct (1).pdf p.1 (0.36)
Latency: retrieval 29 ms | total 3.37 s

Q: What is the capital of Japan?
A: I do not have enough information in the loaded documents to answer that.
Sources: RA10175_CybercrimePreventionAct.pdf p.16 (0.27); RA10175_CybercrimePreventionAct.pdf p.16 (0.23); RA10175_CybercrimePreventionAct.pdf p.12 (0.20)
Latency: retrieval 26 ms | total 0.03 s



In [ ]:
print("=== SCAM-AWARE RAG CHATBOT (information only, not legal advice) ===")
print("Type 'exit' to stop.\n")
while True:
    q = input("You: ").strip()
    if q.lower() in ("exit", "quit", "bye"):
        print("Chatbot: Ingat! Stay safe online.")
        break
    if q:
        show_answer(q)

=== SCAM-AWARE RAG CHATBOT (information only, not legal advice) ===
Type 'exit' to stop.

You: What are common scam tactics?
Q: What are common scam tactics?
A: Money Muling Activities.
Sources: RA12010_AFASA.pdf p.2 (0.47); RA11765_FinancialConsumerProtection.pdf p.7 (0.39); RA12010_AFASA.pdf p.2 (0.39)
Latency: retrieval 19 ms | total 2.47 s

You: What can I do to avoid scams?
Q: What can I do to avoid scams?
A: cybersecurity
Sources: RA11765_FinancialConsumerProtection.pdf p.7 (0.39); RA12010_AFASA.pdf p.2 (0.37); RA12010_AFASA.pdf p.1 (0.34)
Latency: retrieval 20 ms | total 6.32 s

You: exit
Chatbot: Ingat! Stay safe online.


## 4. Evaluation set

Edit these questions to match the documents you actually loaded, and **verify each expected source against the document text**. Keep 20-25 in-scope questions for a meaningful result (add more for the documents you included). A test whose RA number is not among your uploaded file names is skipped automatically.

In [ ]:
# (question, keyword that must appear in the expected source FILE NAME)
IN_SCOPE_TESTS = [
    ("What are the rights of a data subject?", "10173"),
    ("What is the penalty for unauthorized processing of personal information?", "10173"),
    ("What is computer-related fraud under the cybercrime law?", "10175"),
    ("What counts as illegal access under the Cybercrime Prevention Act?", "10175"),
    ("What is cyber-squatting?", "10175"),
    ("What are money muling activities?", "12010"),
    ("What is a social engineering scheme?", "12010"),
    ("What must financial institutions do about a disputed transaction?", "12010"),
    ("Who is required to register their SIM card?", "11934"),
    ("What happens if a SIM card is not registered?", "11934"),
    ("What is the penalty for using a fictitious identity to register a SIM?", "11934"),
    ("What protections do financial consumers have against abusive practices?", "11765"),
]

OUT_OF_SCOPE = [
    "What is the capital of Japan?",
    "How do I cook chicken adobo?",
    "Who won the NBA finals in 2022?",
    "What is the best graphics card for gaming?",
    "How do I fix a flat bicycle tire?",
]

loaded_sources = sorted({c["source"] for c in INDEX["chunks"]})
tests = [(q, k) for q, k in IN_SCOPE_TESTS if any(k in s for s in loaded_sources)]
skipped = [(q, k) for q, k in IN_SCOPE_TESTS if (q, k) not in tests]
print(f"Using {len(tests)} in-scope tests; skipped {len(skipped)} (document not uploaded).")

Using 12 in-scope tests; skipped 0 (document not uploaded).


## 5. Retrieval metrics, abstention threshold and latency

In [ ]:
def hit(results, keyword):
    return [keyword in r["source"] for r in results]

top1 = top3 = 0
rr, lat, in_scores = [], [], []
for q, kw in tests:
    t0 = time.perf_counter()
    res = retrieve(q, top_k=3)
    lat.append((time.perf_counter() - t0) * 1000)
    h = hit(res, kw)
    top1 += h[0]
    top3 += any(h)
    rr.append(1 / (h.index(True) + 1) if any(h) else 0)
    in_scores.append(res[0]["score"])

n = len(tests)
print(f"Top-1 accuracy : {top1 / n:.1%}")
print(f"Top-3 accuracy : {top3 / n:.1%}")
print(f"MRR@3          : {np.mean(rr):.3f}")
print(f"Retrieval latency: mean {np.mean(lat):.1f} ms | p95 {np.percentile(lat, 95):.1f} ms")

oos_scores = [retrieve(q, 1)[0]["score"] for q in OUT_OF_SCOPE]
print(f"\nTop-1 similarity  in-scope : min {min(in_scores):.2f} | median {np.median(in_scores):.2f}")
print(f"Top-1 similarity  out-of-scope: median {np.median(oos_scores):.2f} | max {max(oos_scores):.2f}")
lo, hi = np.percentile(in_scores, 10), np.percentile(oos_scores, 90)
if lo > hi:
    print(f"Suggested MIN_SCORE ~ {(lo + hi) / 2:.2f} (clean separation)")
else:
    print("In-scope and out-of-scope scores overlap: any threshold trades missed answers for wrong ones. Report this as a limitation.")

Top-1 accuracy : 100.0%
Top-3 accuracy : 100.0%
MRR@3          : 1.000
Retrieval latency: mean 26.3 ms | p95 33.8 ms

Top-1 similarity  in-scope : min 0.52 | median 0.67
Top-1 similarity  out-of-scope: median 0.14 | max 0.27
Suggested MIN_SCORE ~ 0.39 (clean separation)


Set `MIN_SCORE` (Section 3) to the value you choose, then evaluate the full chatbot:

In [ ]:
# Full answers for the in-scope set (used for manual grading) + abstention behaviour
rows, totals, answered = [], [], 0
for q, kw in tests:
    r = ask(q)
    totals.append(r["t_total"])
    answered += (not r["abstained"])
    rows.append({
        "question": q, "expected_source": kw, "answer": r["answer"],
        "sources": "; ".join(f"{s['source']} p.{s['page']}" for s in r["sources"]),
        "abstained": r["abstained"], "grade": ""     # fill: correct / partial / hallucinated / wrongly_refused
    })

oos_refused = sum(ask(q)["abstained"] for q in OUT_OF_SCOPE)
print(f"End-to-end latency: mean {np.mean(totals):.2f} s | p95 {np.percentile(totals, 95):.2f} s")
print(f"In-scope questions answered (not refused): {answered}/{len(tests)}")
print(f"Out-of-scope questions correctly refused : {oos_refused}/{len(OUT_OF_SCOPE)}")

grading = pd.DataFrame(rows)
grading.to_csv("grading_sheet.csv", index=False)
grading[["question", "answer", "sources"]]

End-to-end latency: mean 4.46 s | p95 6.39 s
In-scope questions answered (not refused): 12/12
Out-of-scope questions correctly refused : 5/5


,question,answer,sources
0,What are the rights of a data subject?,rights as enumerated in the immediately preced...,RA10173_DataPrivacyAct (1).pdf p.16; RA10173_D...
1,What is the penalty for unauthorized processin...,one (1) year to three (3) years and a fine of ...,RA10173_DataPrivacyAct.pdf p.20; RA10173_DataP...
2,What is computer-related fraud under the cyber...,"unauthorized input, alteration, or deletion of...",RA10175_CybercrimePreventionAct.pdf p.4; RA101...
3,What counts as illegal access under the Cyberc...,the whole or any part of a computer system wit...,RA10175_CybercrimePreventionAct.pdf p.4; RA101...
4,What is cyber-squatting?,The acquisition of a domain name over the inte...,RA10175_CybercrimePreventionAct.pdf p.5; RA101...
5,What are money muling activities?,"obtaining, receiving, depositing, transferring...",RA12010_AFASA.pdf p.2; RA12010_AFASA.pdf p.2; ...
6,What is a social engineering scheme?,a person who obtains ring Schemes,RA12010_AFASA.pdf p.2; RA12010_AFASA.pdf p.2; ...
7,What must financial institutions do about a di...,Coordinated Verification,RA12010_AFASA.pdf p.3; RA12010_AFASA.pdf p.3; ...
8,Who is required to register their SIM card?,all users,RA11934_SIMRegistrationAct.pdf p.3; RA11934_SI...
9,What happens if a SIM card is not registered?,"deactivate, either temporarily or permanently,...",RA11934_SIMRegistrationAct.pdf p.6; RA11934_SI...


### 5.1 Manual answer grading

Retrieval accuracy does not tell you whether the **answer** is right. Open `grading_sheet.csv`, read each answer against the cited page, and fill the `grade` column with one of:

- `correct` — accurate and supported by the cited text
- `partial` — right idea but incomplete or imprecise
- `hallucinated` — contains claims not in the retrieved text
- `wrongly_refused` — the answer exists in your documents but the bot refused

Re-upload the edited file and run the next cell.

In [ ]:
from google.colab import files
files.download("grading_sheet.csv")   # edit it, then upload it back as grading_sheet_graded.csv

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
if os.path.exists("grading_sheet_graded.csv"):
    g = pd.read_csv("grading_sheet_graded.csv")
    counts = g["grade"].value_counts()
    print(counts.to_string())
    print(f"\nFully correct: {counts.get('correct', 0) / len(g):.1%} | correct+partial: {(counts.get('correct', 0) + counts.get('partial', 0)) / len(g):.1%}")
else:
    print("Upload grading_sheet_graded.csv after filling the 'grade' column.")

Upload grading_sheet_graded.csv after filling the 'grade' column.


## 6. Ablation: chunk size and top-k

Retrieval hit-rate (is the expected document among the top-k chunks?) for different chunk sizes. Larger chunks keep more context but dilute the match, and they also overflow the generator's 512-token limit when `top_k` is large.

In [ ]:
rows = []
for cs in [300, 500, 900]:
    idx = build_index(cs, int(cs * 0.15))
    for k in [1, 3, 5]:
        hits = sum(any(hit(retrieve(q, k, idx), kw)) for q, kw in tests)
        rows.append({"chunk_size": cs, "top_k": k, "hit_rate": round(hits / len(tests), 3), "n_chunks": len(idx["chunks"])})
ab = pd.DataFrame(rows)
display(ab.pivot(index="chunk_size", columns="top_k", values="hit_rate"))
ab

top_k,1,3,5
chunk_size,,,
300,1.0,1.0,1.0
500,1.0,1.0,1.0
900,1.0,1.0,1.0


,chunk_size,top_k,hit_rate,n_chunks
0,300,1,1.0,826
1,300,3,1.0,826
2,300,5,1.0,826
3,500,1,1.0,507
4,500,3,1.0,507
5,500,5,1.0,507
6,900,1,1.0,295
7,900,3,1.0,295
8,900,5,1.0,295


## 7. Failure cases worth documenting

Run a few questions that are likely to break the bot and describe what happens (these make strong discussion material):

- A vague question ("Is it illegal?")
- A question that needs two documents ("Which law covers phishing and what must my bank do?")
- A question in Taglish ("Ano ang parusa kapag ginamit ang SIM sa scam?")
- A question whose answer is in a table or list that was split across chunks

In [ ]:
for q in ["Is it illegal?",
          "Which law covers phishing and what must my bank do about it?",
          "Ano ang parusa kapag ginamit ang SIM sa scam?"]:
    show_answer(q)

## 8. Connecting Part 1 and Part 2: entity-driven lookup

Upload `scam_ner_model.zip` from Part 2 and unzip it (`!unzip -q scam_ner_model.zip -d scam_ner_model`). For a suspicious message, the NER model finds the entities, and each entity type triggers a targeted question to the RAG chatbot. This is a prototype of a "scam message explainer", not a scam *detector*.

In [ ]:
!pip -q install spacy
import spacy

ENTITY_QUESTIONS = {
    "IMPERSONATED_BRAND": "What is the offense of pretending to be a bank or company to trick people into giving account details?",
    "URL": "What is phishing or illegal access to accounts under the cybercrime and anti-scamming laws?",
    "EWALLET_ACCOUNT": "What are money muling activities and what is the penalty for using accounts to receive scam money?",
    "PHONE_NO": "What are the rules on SIM registration and the penalties for misusing SIM cards?",
    "AMOUNT": "What can a victim do to dispute and recover money lost in an unauthorized transaction or scam?",
    "URGENCY_PHRASE": "What is a social engineering scheme?",
}

ner = spacy.load("scam_ner_model") if os.path.isdir("scam_ner_model") else None

def analyze_message(msg):
    if ner is None:
        print("Upload and unzip scam_ner_model.zip first.")
        return
    doc = ner(msg)
    print("MESSAGE:", msg, "\n")
    print("Entities found:")
    for e in doc.ents:
        print(f"  [{e.label_}] {e.text}")
    for label in dict.fromkeys(e.label_ for e in doc.ents):
        print(f"\n--- Why '{label}' matters ---")
        show_answer(ENTITY_QUESTIONS[label])

analyze_message("BDO ALERT: Naka-lock ang account mo. Mag-verify agad sa bdo-verify-ph.xyz o ipadala ang P500 sa GCash 09170001111.")

## 9. Write-up template (fill in with *your* numbers)

**Purpose.** A cited question-answering assistant for Philippine anti-scam and cyber-law documents, for students and ordinary users. Information only; not legal advice.

**Document source.** ___ documents (list titles and the date each was downloaded), ___ pages, ___ chunks (chunk size 500 characters, 80 overlap).

**Models.** Embeddings: `all-MiniLM-L6-v2`. Generator: `___` (flan-t5). Retrieval: cosine top-3. Abstention threshold: ___.

**Performance.**
- Retrieval: Top-1 ___, Top-3 ___, MRR@3 ___; mean retrieval latency ___ ms; end-to-end ___ s.
- Answer quality (manual, n = ___): correct ___, partial ___, hallucinated ___, wrongly refused ___.
- Out-of-scope refusal: ___/5. Ablation: best chunk size / top-k = ___.

**Discussion points to consider**
- Where retrieval succeeded or failed (similar laws such as RA 10175 vs RA 12010 compete for the same questions).
- Whether errors come from retrieval or from the small generator; whether `flan-t5-large` helped.
- Taglish questions against English legal text.
- Limitations: small evaluation set, answers graded by one person, legal text changes over time, not legal advice.
- Improvements: a stronger generator, a cross-encoder reranker, section-aware chunking (by Section/Article), query translation for Taglish.